In [119]:
import numpy as np

In [120]:
# Import numpy for numerical operations and array handling
import numpy as np

# Import TensorFlow (deep learning library)
import tensorflow as tf

# Import IMDb dataset from Keras (preloaded movie reviews dataset)
from tensorflow.keras.datasets import imdb

# Import sequence preprocessing utilities (for padding/truncating sequences)
from tensorflow.keras.preprocessing import sequence 

# Import Sequential model (linear stack of layers)
from tensorflow.keras.models import Sequential

# Import layers needed for the model:
# Embedding → for converting word indices to dense vectors
# SimpleRNN → for recurrent neural network layer
# Dense → for fully connected output layer
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


In [121]:
# imdb.load_data(max_features=1000) 

In [122]:
# Set the maximum number of words to keep in the vocabulary
# Only the top 1000 most frequent words will be used, rest will be replaced with '2' (unknown token)
max_features = 1000
max_len = 200

In [123]:
# Load the IMDb dataset (preprocessed movie reviews)
# X_train, X_test → lists of reviews (each review is a list of integers = word indices)
# y_train, y_test → sentiment labels (0 = negative, 1 = positive)
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)


In [124]:
# Print the shape of training data and labels
# X_train.shape → number of reviews in training set
# y_train.shape → number of labels corresponding to training reviews
print(f"Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}")


Training data shape: (25000,), Training labels shape: (25000,)


In [125]:
# Print the shape of test data and labels
print(f"Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}")

Testing data shape: (25000,), Testing labels shape: (25000,)


In [126]:
X_train

array([list([1, 14, 22, 16, 43, 530, 973, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 2, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]),
       list([1, 194, 2, 194, 2, 78, 228, 5, 6, 2, 2, 2, 134, 26, 4, 715, 8, 118, 2, 14, 394, 20, 13, 119, 954, 189,

In [127]:
X_train=np.concatenate((X_train,X_test[:8000]),axis=0)
X_train

y_train=np.concatenate((y_train,y_test[:8000]),axis=0)
y_train

array([1, 0, 0, ..., 0, 1, 1], dtype=int64)

In [128]:
X_test=X_test[8000:]
y_test=y_test[8000:]
y_test



array([1, 1, 1, ..., 0, 0, 0], dtype=int64)

In [129]:
# # Show the first review (as integer indices)
# # Each number represents a word in the top max_features words
# # Special indices:
# # 0 → padding
# # 1 → start-of-sequence token
# # 2 → unknown word (words outside top max_features)
print(X_train[0])

# # Show the label of the first review
# # 0 → negative review
# # 1 → positive review
print(y_train[0])

[1, 14, 22, 16, 43, 530, 973, 2, 2, 65, 458, 2, 66, 2, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 2, 2, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2, 19, 14, 22, 4, 2, 2, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 2, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2, 2, 16, 480, 66, 2, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 2, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 2, 15, 256, 4, 2, 7, 2, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2, 56, 26, 141, 6, 194, 2, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 2, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 2, 88, 12, 16, 283, 5, 16, 2, 113, 103, 32, 15, 16, 2, 19, 178, 32]
1


In [130]:
# Select the first review from the training data
# This review is represented as a list of integers (word indices)
sample_review = X_train[0]


# Select the corresponding label for the first review
# 0 → negative review, 1 → positive review
sample_label = y_train[0]

# Print the label of the review
# 0 → negative, 1 → positive
print(f"Sample label (as integers): {sample_label}")

Sample label (as integers): 1


In [131]:
# Get the dictionary mapping of words to their integer indices from the IMDb dataset
# word_index: {word: index}
# Example: {'the': 1, 'movie': 14, 'was': 16, ...}
word_index = imdb.get_word_index()
word_index 

# Reverse the dictionary to map indices back to words
# This is useful for decoding reviews (X_train) from integers to readable text
# reversed_word_index: {index: word}
# Example: {1: 'the', 14: 'movie', 16: 'was', ...}
reversed_word_index = {value: key for key, value in word_index.items()}


# Display the reversed dictionary
# reversed_word_index


In [132]:
# Decode the integer-encoded review into readable text
# sample_review → list of integers representing words
# reversed_word_index → dictionary mapping index → word
# i-3 → adjust indices because Keras reserves:
#       0 → padding, 1 → start token, 2 → unknown word
# If the index is not found in reversed_word_index, replace with "?" as placeholder
decoded_review = " ".join([reversed_word_index.get(i-3, "?") for i in sample_review])


# Display the decoded review as readable text
decoded_review

"? this film was just brilliant casting ? ? story direction ? really ? the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same ? ? as myself so i loved the fact there was a real ? with this film the ? ? throughout the film were great it was just brilliant so much that i ? the film as soon as it was released for ? and would recommend it to everyone to watch and the ? ? was amazing really ? at the end it was so sad and you know what they say if you ? at a film it must have been good and this definitely was also ? to the two little ? that played the ? of ? and paul they were just brilliant children are often left out of the ? ? i think because the stars that play them all ? up are such a big ? for the whole film but these children are amazing and should be ? for what they have done don't you think the whole story was so ? because it was true and was ? life after all that was ? with us all"

In [133]:
# Import sequence preprocessing utilities from Keras
from tensorflow.keras.preprocessing import sequence


# Set the maximum sequence length
# All reviews will be padded or truncated to this length
max_len = 200


# Pad or truncate training reviews to have the same length = max_len
# padding='pre' by default → pads at the beginning of sequences
# This ensures all input sequences have equal length for RNN input
X_train = sequence.pad_sequences(X_train, maxlen=max_len)


# Pad or truncate test reviews in the same way
X_test = sequence.pad_sequences(X_test, maxlen=max_len)


# Display the padded X_train
X_train


# X_train[0]
print(type(X_train))
print(X_train.dtype)
print(X_train.shape)
print(type(y_train))
print(y_train.dtype)
print(y_train.shape)

<class 'numpy.ndarray'>
int32
(33000, 200)
<class 'numpy.ndarray'>
int64
(33000,)


In [134]:
# Maximum number of words in the vocabulary
# This should match the vocabulary size used in preprocessing / tokenizer
max_features = 1000


# Maximum length of each input sequence (after padding/truncating)
max_len = 200


# Initialize a Sequential model (a linear stack of layers)
model = Sequential()
# 1️⃣ Embedding layer
# Converts integer-encoded words into dense vectors of fixed size (128)
# max_features → size of vocabulary (input dimension)
# 128 → size of the embedding vector for each word (output dimension)
# input_length → length of input sequences
model.add(Embedding(max_features, 128, input_length=max_len))


# 2️⃣ SimpleRNN layer
# Processes the sequence of embeddings to capture sequential relationships
# 128 → number of units in the RNN layer
# activation='tanh' → non-linear activation function for RNN units
model.add(SimpleRNN(128, activation='tanh'))


# 3️⃣ Dense output layer
# Single neuron output layer for binary classification (positive/negative review)
# activation='sigmoid' → outputs probability between 0 and 1
model.add(Dense(1, activation="sigmoid"))


# Build the model by specifying the input shape
# input_shape = (batch_size, sequence_length)
# None → allows any batch size
# max_len → length of each input sequence (number of words)
# number of samples given to the model at once
model.build(input_shape=(None, max_len))


# Print a summary of the model
# Shows each layer, output shape, and number of parameters
model.summary()


model.compile(optimizer='adam',loss="binary_crossentropy",metrics=["accuracy"])


Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, 200, 128)          128000    
                                                                 
 simple_rnn_5 (SimpleRNN)    (None, 128)               32896     
                                                                 
 dense_5 (Dense)             (None, 1)                 129       
                                                                 
Total params: 161025 (629.00 KB)
Trainable params: 161025 (629.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [135]:
#create an instance of EarlyStopping Callbacks
from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
earlystopping


##tain the model with early stopping
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[earlystopping]
)




Epoch 1/10
825/825 [==============================] - 83s 98ms/step - loss: 0.6421 - accuracy: 0.6093 - val_loss: 0.5758 - val_accuracy: 0.6912
Epoch 2/10
825/825 [==============================] - 74s 90ms/step - loss: 0.6455 - accuracy: 0.6275 - val_loss: 0.6635 - val_accuracy: 0.5792
Epoch 3/10
825/825 [==============================] - 68s 82ms/step - loss: 0.6299 - accuracy: 0.6233 - val_loss: 0.6194 - val_accuracy: 0.6386
Epoch 4/10
825/825 [==============================] - 63s 76ms/step - loss: 0.5893 - accuracy: 0.6859 - val_loss: 0.6579 - val_accuracy: 0.5806
Epoch 5/10
825/825 [==============================] - 61s 74ms/step - loss: 0.5164 - accuracy: 0.7462 - val_loss: 0.4967 - val_accuracy: 0.7706
Epoch 6/10
825/825 [==============================] - 66s 80ms/step - loss: 0.5173 - accuracy: 0.7479 - val_loss: 0.5136 - val_accuracy: 0.7618
Epoch 7/10
825/825 [==============================] - 66s 79ms/step - loss: 0.5015 - accuracy: 0.7644 - val_loss: 0.5901 - val_accuracy:

In [136]:
##Save model file 
model.save("simple_rn_imdb.h5")

c:\Users\HP\.conda\envs\newenv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
